# UrbanSound8K reproducible training — colab-training
Select **Runtime → Change runtime type → GPU** when CUDA training is wanted. This notebook is only a controller: training, mixed-duration validation, augmentation, calibration and export stay in the repository CLI.

TensorFlow 2.15/Keras 2.15 are retained using a separate Python 3.11 environment. Colab's notebook kernel and preinstalled TensorFlow are left alone. Dataset files must already be in Drive; no dataset download or test-fold evaluation runs automatically. Final experiment directories are allocated on Drive; checkpoints/converter work and caches stay in `/content`. Each subprocess releases its GPU memory when it exits.


In [ ]:
#@title Mount Drive and check the requested branch
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, json, shutil

drive.mount('/content/drive')
REPOSITORY = 'https://github.com/FleabagEE/embedded-systme-project.git'
BRANCH = 'colab-training'
REPO_DIR = Path('/content/embedded-systme-project')
remote = subprocess.run(['git', 'ls-remote', '--exit-code', '--heads', REPOSITORY, f'refs/heads/{BRANCH}'], capture_output=True, text=True)
if remote.returncode != 0 or not remote.stdout.strip():
    raise RuntimeError(f'Required branch {BRANCH!r} does not exist or cannot be accessed. No fallback to main. {remote.stderr}')
if REPO_DIR.exists():
    current = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
    origin = subprocess.check_output(['git', '-C', str(REPO_DIR), 'remote', 'get-url', 'origin'], text=True).strip()
    if current != BRANCH or dirty or origin != REPOSITORY:
        raise RuntimeError('Existing checkout has another branch, origin, or local edits. Choose a clean /content checkout; nothing was reset.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
hardware = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
print('CUDA hardware:', hardware.stdout.strip() if hardware and hardware.returncode == 0 else 'not detected; CPU fallback available')


In [ ]:
#@title Install the project in an isolated Python 3.11 training environment
# The existing notebook kernel is not downgraded. No credentials are embedded.
ENV_DIR = Path('/content/urbansound-tf215')
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv>=0.9,<1'], check=True)
if not (ENV_DIR / 'bin/python').exists():
    subprocess.run(['uv', 'venv', '--python', '3.11', str(ENV_DIR)], check=True)
TRAINING_PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', TRAINING_PYTHON, '-r', 'requirements-colab.txt'], check=True)
# Make pip-provided CUDA/cuDNN libraries visible to child training processes.
site = Path(subprocess.check_output([TRAINING_PYTHON, '-c', 'import site; print(site.getsitepackages()[0])'], text=True).strip())
CUDA_LIB_DIRS = sorted(str(p) for p in (site / 'nvidia').glob('*/lib'))
TRAIN_ENV = os.environ.copy()
TRAIN_ENV['LD_LIBRARY_PATH'] = ':'.join(CUDA_LIB_DIRS + [TRAIN_ENV.get('LD_LIBRARY_PATH', '')])
TRAIN_ENV['TMPDIR'] = '/content/urbansound-work'
Path(TRAIN_ENV['TMPDIR']).mkdir(parents=True, exist_ok=True)
probe_code = 'import json,tensorflow as tf; print(json.dumps({"tensorflow":tf.__version__,"cuda_build":tf.test.is_built_with_cuda(),"gpus":[d.name for d in tf.config.list_physical_devices("GPU")]}))'
probe = subprocess.run([TRAINING_PYTHON, '-c', probe_code], env=TRAIN_ENV, text=True, capture_output=True, check=True)
print(probe.stdout)
print(probe.stderr)
info = json.loads(probe.stdout.strip().splitlines()[-1])
if hardware and hardware.returncode == 0 and not info['gpus']:
    raise RuntimeError('NVIDIA hardware is present but TensorFlow cannot use it. Inspect the CUDA/driver setup above; CPU is not forced as a workaround.')


In [ ]:
#@title Dataset and artifact locations (independent)
dataset_drive_path = '/content/drive/MyDrive/EmbeddedSystem/urbansound8k' #@param {type:"string"}
output_path = '/content/drive/MyDrive/EmbeddedSystem/models' #@param {type:"string"}
stage_dataset_to_content = True #@param {type:"boolean"}
source = Path(dataset_drive_path)
if not (source / 'metadata/UrbanSound8K.csv').is_file() or not (source / 'audio').is_dir():
    raise FileNotFoundError('Expected metadata/UrbanSound8K.csv and audio/fold*/ in dataset_drive_path. No automatic download.')
DATASET_ROOT = source
if stage_dataset_to_content:
    import hashlib
    fingerprint = hashlib.sha256((str(source) + str((source / 'metadata/UrbanSound8K.csv').stat().st_mtime_ns)).encode()).hexdigest()[:12]
    DATASET_ROOT = Path('/content/datasets') / fingerprint
    ready = DATASET_ROOT / '.copy_complete'
    if not ready.exists():
        shutil.copytree(source, DATASET_ROOT, dirs_exist_ok=True)
        ready.write_text('complete')
OUTPUT_ROOT = Path(output_path)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET_ROOT, '\nFinalized experiments:', OUTPUT_ROOT)


In [ ]:
#@title Experiment controls
training_duration = 'mixed' #@param ["1.0", "1.5", "2.0", "mixed"]
mixed_durations = '[1.0, 2.0]' #@param {type:"string"}
mel_bins = 64 #@param [40, 64]
width_multiplier = 'all' #@param ["0.25", "0.5", "1.0", "all"]
conv_type = 'SC' #@param ["SC", "DW"]
augmentation_enabled = True #@param {type:"boolean"}
deterministic = True #@param {type:"boolean"}
seed = 42 #@param {type:"integer"}
epochs = 50 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
experiment_mode = 'single' #@param ["single", "batch"]
skip_completed = True #@param {type:"boolean"}
# Batch choices can include scalar durations or mixed subsets, e.g. [[1.0,1.5], [1.5,2.0]].
durations = [1.0, 1.5, 2.0]
mel_bin_choices = [40, 64]
widths = [0.25, 0.5, 1.0]
conv_types = ['SC', 'DW']

def width_token(value):
    return {'0.25':'25', '0.5':'5', '1.0':'1', '1':'1', 'all':'all'}[str(value)]
selected = json.loads(mixed_durations) if training_duration == 'mixed' else [float(training_duration)]
if not selected or len(set(selected)) != len(selected) or any(d not in (1.0,1.5,2.0) for d in selected):
    raise ValueError('Choose unique durations from 1.0, 1.5, 2.0')
common = {
    'dataset.root': str(DATASET_ROOT), 'dataset.cache_dir': '/content/urbansound-cache',
    'runtime.output_dir': str(OUTPUT_ROOT), 'runtime.work_dir': '/content/urbansound-work',
    'runtime.device': 'auto', 'runtime.deterministic': deterministic, 'training.seed': seed, 'training.epochs': epochs,
    'training.batch_size': batch_size, 'augmentation.enabled': augmentation_enabled,
    'quantization.output_duration_secs': []}


In [ ]:
#@title Train through the repository CLI
command = [TRAINING_PYTHON, '-m', 'training.train' if experiment_mode == 'single' else 'training.sweep', '--config', 'config.yaml']
settings = dict(common)
if experiment_mode == 'single':
    settings.update({'training.duration_secs': selected, 'audio.n_mels': mel_bins,
                     'model.width': width_token(width_multiplier), 'model.conv_type': conv_type})
else:
    matrix = {'parameters': {
        'training.duration_secs': [d if isinstance(d, list) else [d] for d in durations],
        'audio.n_mels': mel_bin_choices, 'model.width': [width_token(w) for w in widths],
        'model.conv_type': conv_types}}
    space_path = Path('/content/urbansound-work/sweep.json')
    space_path.write_text(json.dumps(matrix))
    command += ['--space', str(space_path)]
for key, value in settings.items():
    command += ['--set', f'{key}={json.dumps(value)}']
if skip_completed:
    command.append('--skip-completed')
print(' '.join(command))
outcome = subprocess.run(command, cwd=REPO_DIR, env=TRAIN_ENV, check=False)
if outcome.returncode:
    print('One or more experiments failed. Valid failure reports and earlier completed artifacts remain on Drive; inspect the table below.')


In [ ]:
#@title Concise summary from completed/failed reports
import csv, html
from IPython.display import display, HTML
subprocess.run([TRAINING_PYTHON, '-m', 'training.registry', '--model-dir', str(OUTPUT_ROOT)], cwd=REPO_DIR, env=TRAIN_ENV, check=True)
with (OUTPUT_ROOT / 'summary.csv').open() as stream:
    rows = list(csv.DictReader(stream))
columns = ['experiment_id', 'status', 'durations', 'conv_type', 'width', 'best_epoch', 'val_accuracy', 'macro_f1']
table = '<table><tr>' + ''.join(f'<th>{html.escape(c)}</th>' for c in columns) + '</tr>'
for row in rows:
    table += '<tr>' + ''.join(f'<td>{html.escape(row.get(c, ""))}</td>' for c in columns) + '</tr>'
display(HTML(table + '</table>'))
for row in rows:
    print(row['experiment_id'], row['status'], row['report_path'])
    print('Exported artifacts:', row.get('exports', '{}'))


## Optional final held-out evaluation
Run only after configuration/checkpoint selection is finished. The default is off. Test metrics are recorded separately and never used by training callbacks or the batch sweep.


In [ ]:
run_final_test = False #@param {type:"boolean"}
final_experiment_id = '' #@param {type:"string"}
if run_final_test:
    import re
    if not re.fullmatch(r'dscnn_[0-9]+', final_experiment_id):
        raise ValueError('Enter one completed experiment ID, e.g. dscnn_001')
    experiment = OUTPUT_ROOT / final_experiment_id
    report = json.loads((experiment / 'report.json').read_text())
    if report['state']['status'] != 'complete':
        raise ValueError('Final evaluation requires a completed experiment')
    final_durations = report['config']['training']['duration_secs'] or [report['config']['audio']['duration_sec']]
    for duration in final_durations:
        subprocess.run([TRAINING_PYTHON, '-m', 'evaluation.evaluate', '--model-dir', str(experiment),
                        '--split', 'test', '--duration', str(duration)], cwd=REPO_DIR, env=TRAIN_ENV, check=True)
